# AIC2026 — Vòng thử nghiệm, chạy ĐẦY ĐỦ trên Kaggle (có QA đọc ảnh)

Notebook này làm đúng một việc mà máy local không làm được: **trả lời 3 câu Q&A bằng VLM
đọc ảnh keyframe thật**. Tầng tìm (KIS/TRAKE) chạy y hệt local và phải ra **cùng kết quả
từng byte** — có một ô kiểm đúng điều đó.

## Trước khi chạy — 3 việc bắt buộc

1. **UPLOAD LẠI DATASET CODE.** Bản đang có trên Kaggle là bản *trước* 20/08, tức vẫn còn
   nguyên 5 lỗi đã sửa hôm nay — trong đó có lỗi làm cả 21 câu KIS/QA đi tìm bằng chuỗi
   `"query-p1-1-kis"`. Nén thư mục `D:\COMP\AIC2026\tự làm\system thử\` rồi tạo
   **New Version** cho dataset code. Nhớ **giữ cả `data/` và `out/`**: `data/` chứa đề đã
   chuyển sang JSON + file ghim; `out/viet_lai_truy_van.json` là cache LLM viết lại —
   không có nó thì tốn thêm request và **kết quả không còn khớp bản local**.
2. **Add đủ dataset:** code · `aic2026_data_nhe` · `aic2026_siglip2_so400m` ·
   `aic2026_siglip2` · và bộ **keyframes** (`data-batch-1-aic`). Cái cuối mới là thứ local
   không có.
3. **Bật Internet** (Settings → Internet on) cho Gemini. **GPU không cần** — tầng tìm chạy
   CPU, chỉ chậm hơn chút ở khâu mã hoá câu chữ.

## Ô 1 — Nâng cấp SDK Gemini, RỒI RESTART KERNEL

🔴 Kaggle cài sẵn `google-genai` **< 2.0.0**. Với bản đó server trả
`400 legacy Interactions API schema is no longer supported` cho **mọi** lời gọi Gemini — và
thông báo đọc y hệt lỗi hết quota, nên rất dễ đi sai hướng cả tiếng đồng hồ. Đúng lỗi này
đã bắt được trong buổi diễn tập 19/08.

Chạy ô này, đợi xong, rồi bấm **Run → Restart & clear cell outputs**, sau đó chạy tiếp từ
Ô 2. Không restart thì bản cũ vẫn nằm trong bộ nhớ tiến trình.

In [ ]:
!pip install -q -U "google-genai>=2.3.0"
import google.genai as g
print("google-genai:", g.__version__, "  <- phải >= 2.3.0")
print("\n>>> BÂY GIỜ BẤM Run -> Restart & clear cell outputs <<<")

## Ô 2 — Tìm code, chép sang thư mục làm việc, kiểm ĐÚNG BẢN

In [ ]:
import os, shutil, pathlib

def thu_muc(goc, sau, bo_qua=('keyframes',)):
    """Thư mục con tới độ sâu `sau`. Không chui vào `bo_qua` — tầng ảnh, hàng vạn file."""
    muc, ra = [goc], []
    for _ in range(sau):
        tiep = []
        for d in muc:
            if os.path.basename(d) in bo_qua:
                continue
            try:
                tiep += [e.path for e in os.scandir(d) if e.is_dir()]
            except OSError:
                pass
        ra += tiep
        muc = tiep
    return ra

DIRS = thu_muc('/kaggle/input', 6)
goc = next((d for d in DIRS if os.path.exists(os.path.join(d, 'nop_bai.py'))), None)
assert goc, "Không thấy dataset code (dấu hiệu nhận biết: có file nop_bai.py)"

shutil.copytree(goc, '/kaggle/working/sys', dirs_exist_ok=True)
os.chdir('/kaggle/working/sys')
print("nguồn code:", goc, "\n")

kiem = {
    "lớp ghim  aic2026/core/ghim.py":       pathlib.Path('aic2026/core/ghim.py').exists(),
    "đề JSON   data/de_thunghiem.json":     pathlib.Path('data/de_thunghiem.json').exists(),
    "file ghim data/ghim_thunghiem.json":   pathlib.Path('data/ghim_thunghiem.json').exists(),
    "cache LLM out/viet_lai_truy_van.json": pathlib.Path('out/viet_lai_truy_van.json').exists(),
    "reset() trong core/retrieval.py":      'def reset' in pathlib.Path(
        'aic2026/core/retrieval.py').read_text(encoding='utf-8'),
}
for k, v in kiem.items():
    print(("  ✅ " if v else "  ❌ ") + k)
assert all(kiem.values()), ("Dataset code là BẢN CŨ hoặc thiếu data/out. Upload lại rồi "
                            "chạy lại từ đầu — chạy tiếp là phí một buổi.")

## Ô 3 — Khoá API

Đặt trước ở **Add-ons → Secrets**, đúng tên `GEMINI_API_KEY`. Chuỗi dự phòng Cloudflare
cần thêm `CLOUDFLARE_API_TOKEN` + `CLOUDFLARE_ACCOUNT_ID`; thiếu cũng được, vì Gemini đã
xếp chồng 5 model.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
uc = UserSecretsClient()
for ten, bat_buoc in [("GEMINI_API_KEY", True),
                      ("CLOUDFLARE_API_TOKEN", False),
                      ("CLOUDFLARE_ACCOUNT_ID", False)]:
    try:
        os.environ[ten] = uc.get_secret(ten)
        print(f"  ✅ {ten}: {len(os.environ[ten])} ký tự")
    except Exception:
        print(f"  {'❌' if bat_buoc else '—'} {ten}: không có")
        if bat_buoc:
            raise

## Ô 4 — Dựng `config_kaggle.yaml` từ đường dẫn thật

Khác bản local đúng một khoá: **`qa.image_roots`** trỏ vào các thư mục keyframes. Đó là
toàn bộ lý do phải lên Kaggle.

In [ ]:
def tim(ten):
    return sorted(d for d in DIRS if os.path.basename(d) == ten)

nhe     = tim('aic2026_data_nhe')
so400m  = tim('aic2026_siglip2_so400m')
base    = tim('aic2026_siglip2')
kf      = tim('keyframes')

for ten, v in [('data_nhe', nhe), ('so400m', so400m), ('base-224', base)]:
    print(f"  {ten:9s}", v[0] if v else "❌ KHÔNG THẤY")
print(f"  keyframes {len(kf)} thư mục")

assert nhe and so400m and base, "Thiếu dataset vector"
assert kf, ("❌ Chưa add dataset keyframes. Không có ảnh thì QA vẫn rơi về đáp án rẻ, "
            "y hệt bản local — cả notebook này thành vô nghĩa.")

cfg = ("extends: config.example.yaml\n"
       "offline:\n"
       f"  btc_data_root: {nhe[0]}\n"
       f"  image_feature_root: {so400m[0]}\n"
       "retrieval:\n  hop_nhat:\n    phu:\n"
       f"    - image_feature_root: {base[0]}\n"
       "qa:\n  image_roots:\n"
       + "".join(f"  - {p}\n" for p in kf))
open('config_kaggle.yaml', 'w', encoding='utf-8').write(cfg)
print("\n" + cfg)

## Ô 5 — Kiểm resolver ảnh TRƯỚC khi đốt quota

10 giây, **0 request**. Nó trả lời đúng một câu: với `(video, số keyframe)` bất kỳ, có tìm
ra đúng file `.jpg` không. Bước này hỏng thì mọi request Gemini sau đó là tiền vứt đi —
đường QA sẽ ném lỗi thiếu ảnh rồi âm thầm hạ xuống đáp án rẻ.

In [ ]:
!python -X utf8 kiem_resolver_anh.py --config config_kaggle.yaml

## Ô 6 — Kiểm khoá API (1 request mỗi provider)

In [ ]:
!python -X utf8 kiem_khoa_api.py --config config_kaggle.yaml

## Ô 7 — BÀI NỘP L3: thuần máy, **có QA đọc ảnh**

Khác `aic2026thunghiemL1may.zip` ở đúng một chỗ: **bỏ cờ `--qa-cheap`**, nên 3 câu Q&A
được Gemini đọc top-10 ảnh thật thay vì trả `"chợ"` / `"nhà bếp"`.

Quota: `batch_max_images: 10` gộp 10 ảnh thành **1 request/câu** → 3 câu = **3 request**,
thoải mái trong RPD 20 của bậc flash.

⚠ **Đọc hai thứ trong output:** dòng `đáp án phân biệt` của mỗi câu QA, và mọi dòng
`⚠ VLM lỗi -> đã hạ xuống QA rẻ`. Thấy dòng thứ hai nghĩa là đáp án **không phải** của
Gemini — mà nhìn vào zip thì không có dấu hiệu nào để biết.

In [ ]:
!python -X utf8 nop_bai.py \
    --config config_kaggle.yaml \
    --truy-van data/de_thunghiem.json \
    --ra /kaggle/working/aic2026thunghiemL3vlm.zip

## Ô 8 — BÀI NỘP L4: thêm 3 ghim của người vận hành

Tương đương `aic2026thunghiemL2ghim.zip` nhưng QA đã có VLM. **Đây là bản mạnh nhất**, để
dành nộp lần cuối.

In [ ]:
!python -X utf8 nop_bai.py \
    --config config_kaggle.yaml \
    --truy-van data/de_thunghiem.json \
    --ghim data/ghim_thunghiem.json \
    --ra /kaggle/working/aic2026thunghiemL4vlmghim.zip

## Ô 9 — Đối chiếu với bản local

Kaggle và local dùng cùng vector, cùng cache viết lại, nên tầng tìm phải cho kết quả
**giống hệt**. Ô này kiểm điều đó và cô lập đúng phần *thật sự* thay đổi: 3 file QA.

Có file KIS/TRAKE nào lệch → dừng, điều tra trước khi nộp.

In [ ]:
import zipfile
cu  = zipfile.ZipFile('out/NOP/aic2026thunghiemL1may.zip')      # dựng ở local, đi kèm dataset
moi = zipfile.ZipFile('/kaggle/working/aic2026thunghiemL3vlm.zip')

lech = [n for n in sorted(cu.namelist()) if cu.read(n) != moi.read(n)]
print(f"{len(lech)}/24 file khác bản local:")
for n in lech:
    print("   ", n)

qa   = {n for n in lech if n.endswith('-qa.csv')}
khac = set(lech) - qa
print()
print(f"  QA đổi (ĐÚNG KỲ VỌNG): {len(qa)}/3")
print("  ✅ tầng tìm tái lập y hệt local" if not khac else
      f"  ❌ {len(khac)} file KIS/TRAKE LỆCH — điều tra trước khi nộp: {sorted(khac)}")

## Ô 10 — Xem đáp án QA thật sự sẽ nộp

In [ ]:
import zipfile, csv, io
z = zipfile.ZipFile('/kaggle/working/aic2026thunghiemL4vlmghim.zip')
for n in sorted(x for x in z.namelist() if x.endswith('-qa.csv')):
    rows = list(csv.reader(io.StringIO(z.read(n).decode('utf-8'))))
    dap = []
    for r in rows:
        a = ",".join(r[2:])
        if a not in dap:
            dap.append(a)
    print(f"--- {n}   {len(rows)} dòng, {len(dap)} đáp án khác nhau")
    print(f"    hạng 1: {rows[0][0]}  frame {rows[0][1]}")
    for a in dap[:4]:
        print(f"    đáp án: {a[:90]}")
    print()

## Ô 11 — Soi lại bằng metadata (0 request)

In tiêu đề video của các dòng đầu mỗi câu — cách duy nhất tự đánh giá khi chưa có đáp án.
Xem §5 của `BAO_CAO_VONG_THU_NGHIEM.md`.

In [ ]:
!python -X utf8 soi_ket_qua.py /kaggle/working/aic2026thunghiemL4vlmghim.zip \
    --config config_kaggle.yaml --de data/de_thunghiem.json --sau 3 \
    --ra /kaggle/working/soi_L4.md 2>&1 | tail -45

## Ô 12 — Sao lưu cache rồi tải file về

`out/` chứa cache VLM và cache viết lại truy vấn. **Giữ lại**: câu đã hỏi sẽ không tốn
quota lần hai ở phiên sau. Tải `out_cache_backup.zip` về, giải nén đè lên `out/` ở local,
rồi upload kèm dataset lần tới.

Hai file nộp bài lấy ở panel **Output** bên phải.

In [ ]:
!cd /kaggle/working/sys && zip -qr /kaggle/working/out_cache_backup.zip out
!ls -la /kaggle/working/*.zip /kaggle/working/*.md

---
# PHẦN 2 — ĐƯA MẮT NGƯỜI VÀO VÒNG LẶP

Đây là phần đáng giá nhất, và số đo nói rõ vì sao:

| | |
|---|---|
| `R@100` trên dev set | **57,5%** — khung đúng ĐÃ nằm trong 100 dòng máy trả về |
| điểm public thật (20/08) | **0,467/câu** — nhưng chỉ ăn chừng này |

Chênh lệch đó **không phải** vì mô hình yếu. Nó nằm ở chỗ khung đúng không đứng
hạng 1, và `Final = (1/5)·Σ_{k∈{1,5,20,50,100}} R@k` phạt rất nặng chuyện đó:
cùng một khung, hạng 100 được **0,2** điểm, hạng 1 được **1,0**.

Máy không sửa được. Mắt người thì sửa trong ba giây.

## Ô 13 — Sinh trang xem ảnh

Ảnh được nhúng thẳng vào HTML dạng base64, nên tải về rồi **mở offline vẫn xem
được** — không cần Kaggle sống, không cần mạng. Cỡ ~280 KB mỗi câu ở `--sau 100`.

Chạy hết 24 câu × 100 ảnh mất vài phút (đọc 2400 file JPEG từ `/kaggle/input`).
Muốn thử nhanh một câu thì thêm `--chi query-p1-2-kis`.

In [ ]:
!python -X utf8 xem_anh.py /kaggle/working/aic2026thunghiemL4vlmghim.zip \
    --config config_kaggle.yaml \
    --de data/de_thunghiem.json \
    --sau 100 \
    --ra /kaggle/working/xem

!cd /kaggle/working && zip -qr xem.zip xem && ls -la xem.zip

## Ô 14 — (làm ở máy bạn) xem và chọn

1. Tải `xem.zip` từ panel **Output**, giải nén, mở `index.html`.
2. Mở từng câu. **Bấm vào ảnh đúng.** Bấm nhiều ảnh cũng được — thứ tự bấm chính
   là thứ tự dòng nộp, nên bấm cái chắc nhất trước.
3. Bấm **Sao chép** ở khay dưới cùng, dán vào `data/ghim.json`.

Ưu tiên soi những câu mà `soi_ket_qua.py` báo tiêu đề video hạng 1 không ăn nhập
— với bộ đề thử nghiệm đó là các câu **1, 2, 8, 14, 21, 23** (mẩu tin ngắn trong
bản tin 60 Giây). Đó là chỗ máy chắc chắn sai và mắt người có lợi thế lớn nhất.

Ngược lại, các câu **5, 6, 9, 10, 11, 12, 17, 20** máy đã chọn đúng video rồi;
ở đó việc của bạn nhẹ hơn: chỉ cần chỉ ra **đúng khung** trong số các khung cùng
video, và cái đó cũng đáng cả 0,4–0,8 điểm mỗi câu.

File ghim sau khi dán trông như thế này — `khung` và `videos` dùng chung được:

```json
{
  "query-p1-21-kis": {
    "khung": ["L22_V014|18327", "L22_V014|18402"],
    "ghi_chu": "nhìn thấy phòng lab EPFL ở khung này"
  },
  "query-p1-19-qa": {
    "videos": ["L24_V011"], "so_dong": 100,
    "answer": "Hỏa hồng Nhựt Tảo oanh thiên địa, Kiếm bạt Kiên Giang khấp quỷ thần"
  }
}
```

## Ô 15 — Dựng lại bài nộp với lựa chọn của bạn

Upload `ghim.json` đã sửa (kèm dataset, hoặc dán thẳng bằng ô bên dưới), rồi chạy.

Khung người chọn được đặt lên **trước tất cả**, kể cả trước video được ghim; bài
của máy vẫn nằm nguyên phía sau làm lưới an toàn, nên chọn nhầm cũng không mất
hết.

In [ ]:
# Cách nhanh: dán thẳng JSON vào đây thay vì upload lại dataset.
GHIM = r"""
{
  "_ghi_chu": "dán từ trang xem ảnh",

  "query-p1-19-qa": {
    "videos": ["L24_V011"],
    "so_dong": 100,
    "answer": "Hỏa hồng Nhựt Tảo oanh thiên địa, Kiếm bạt Kiên Giang khấp quỷ thần",
    "ghi_chu": "video duy nhất trong 873 video có 'Nguyễn Trung Trực' ở tiêu đề"
  },
  "query-p1-4-trake":  {"videos": ["L26_V194"], "so_dong": 50},
  "query-p1-20-kis":   {"videos": ["L26_V004"], "so_dong": 50}
}
"""
import json, pathlib
d = json.loads(GHIM)                      # nổ ngay tại đây nếu JSON sai cú pháp
pathlib.Path('data/ghim_nguoi.json').write_text(GHIM, encoding='utf-8')
print(f"{len([k for k in d if not k.startswith('_')])} truy vấn được ghim")

In [ ]:
!python -X utf8 nop_bai.py \
    --config config_kaggle.yaml \
    --truy-van data/de_thunghiem.json \
    --ghim data/ghim_nguoi.json \
    --ra /kaggle/working/aic2026thunghiemL6nguoi.zip

## Ô 16 — Kiểm lần cuối trước khi nộp

Đếm xem lựa chọn của bạn có thật sự lên đầu không. Đừng nộp nếu ô này báo lệch.

In [ ]:
import zipfile, csv, io, json
g = json.loads(open('data/ghim_nguoi.json', encoding='utf-8').read())
z = zipfile.ZipFile('/kaggle/working/aic2026thunghiemL6nguoi.zip')
print(f"{len(z.namelist())} file · gốc zip = "
      f"{sorted({n.split('/')[0] for n in z.namelist()})}\n")

for qid, muc in g.items():
    if qid.startswith('_'):
        continue
    rows = list(csv.reader(io.StringIO(
        z.read(f'submission/{qid}.csv').decode('utf-8'))))
    print(f"{qid}: {len(rows)} dòng · hạng 1 = {rows[0][0]} / {','.join(rows[0][1:2])}")
    for k in (muc.get('khung') or [])[:5]:
        v, f = (k.split('|') if isinstance(k, str) else [str(k[0]), str(k[1])])
        hang = next((i for i, r in enumerate(rows, 1)
                     if r[0].upper() == v.upper() and r[1] == f), None)
        print(f"    khung {v}/{f} -> hạng {hang}"
              + ("  ❌ KHÔNG CÓ TRONG BÀI NỘP" if hang is None else
                 "  ✅" if hang <= len(muc.get('khung') or []) else "  ⚠ không ở đầu"))

---
# PHẦN 3 — VLM XẾP LẠI TOP-100 *(thay cho việc bắt người nhìn ảnh)*

Cùng một khe hở như Phần 2, nhưng **tự động**:

```
R@100 = 57,5%   khung đúng ĐÃ nằm trong 100 dòng
điểm  = 46,7%   nhưng nó không đứng hạng 1
```

Khác biệt: con mắt nhìn 100 tấm ảnh là **Gemini**, không phải bạn. Nên nó đo
được, tái lập được, và chạy được trong 3 tiếng của đợt 1 mà không cần bạn bấm
tấm nào.

**Cơ sở, không phải hy vọng.** `thu_vlm_cham_anh.py` đo trên 120 ảnh dev có
ground truth thật (21/08, chạy local):

| model | mô tả | top-1 |
|---|---|---|
| `gemini-3.5-flash-lite` | tiếng Anh | **10/10** |
| `gemini-3.5-flash-lite` | **tiếng Việt** | **9/10** |
| *(đoán mò)* | | *1/10* |

⚠ Phép thử đó là chế độ DỄ — 10 ảnh từ 10 video khác nhau. Ở đây 100 ứng viên
đều là top của encoder nên trông na ná nhau. Vì vậy mặc định là `--che-do rrf`
(VLM chỉ góp phiếu cùng hạng gốc) chứ không phải `diem` (VLM toàn quyền).

## Ô 17 — Chạy rerank

Chi phí: 10 request/câu × 21 câu KIS+QA = **~210 request**, ~6s mỗi request →
**khoảng 20 phút**. Bậc lite có RPD 500 nên thừa quota. Cache ghi sau MỖI câu,
nên quota chết giữa chừng cũng không mất phần đã chạy — chạy lại là đi tiếp.

TRAKE được chép nguyên, không rerank (một dòng TRAKE là 4 khung phải khớp 4 sự
kiện theo thứ tự — cần prompt và phép đo khác hẳn).

In [ ]:
!python -X utf8 chay_rerank.py \
    /kaggle/working/aic2026thunghiemL4vlmghim.zip \
    --config config_kaggle.yaml \
    --de data/de_thunghiem.json \
    --sau 100 --che-do rrf \
    --ra /kaggle/working/aic2026thunghiemL7rerank.zip

## Ô 18 — Rerank đã đổi những gì

Đọc kỹ ô này trước khi nộp. Ba thứ cần thấy:

* **`hạng 1 đổi`** — bao nhiêu câu bị đổi dòng đầu. Bằng 0 nghĩa là VLM không
  cãi encoder câu nào (đáng ngờ). Bằng gần hết nghĩa là nó cãi mọi câu (cũng
  đáng ngờ). Khoảng 30–70% là hợp lý.
* **`lô hoà điểm`** — lô mà mọi ảnh cùng một điểm. Nhiều = model không nhìn ảnh,
  đúng cái bẫy đã bắt được ở `gemini-3.6-flash` ngày 20/08.
* **video hạng 1** — có đổi sang video khác không, hay chỉ đổi khung trong cùng
  video. Đổi khung trong đúng video là thứ ăn điểm chắc nhất.

In [ ]:
import zipfile, csv, io
cu  = zipfile.ZipFile('/kaggle/working/aic2026thunghiemL4vlmghim.zip')
moi = zipfile.ZipFile('/kaggle/working/aic2026thunghiemL7rerank.zip')

def dau(z, n):
    r = list(csv.reader(io.StringIO(z.read(n).decode('utf-8'))))
    return r[0][0], r[0][1]

doi_khung = doi_video = giu = 0
for n in sorted(cu.namelist()):
    if 'trake' in n:
        continue
    (v1, f1), (v2, f2) = dau(cu, n), dau(moi, n)
    if (v1, f1) == (v2, f2):
        giu += 1
        continue
    if v1 == v2:
        doi_khung += 1
        print(f"  {n[11:-4]:22s} cùng video {v1}: khung {f1} -> {f2}")
    else:
        doi_video += 1
        print(f"  {n[11:-4]:22s} ĐỔI VIDEO {v1}/{f1} -> {v2}/{f2}")

print(f"\n  giữ nguyên hạng 1 : {giu}")
print(f"  đổi KHUNG cùng video: {doi_khung}   <- ăn điểm chắc nhất")
print(f"  đổi hẳn VIDEO       : {doi_video}   <- rủi ro cao nhất")

## Ô 19 — Nộp và so số

Vòng thử nghiệm còn **1 lần nộp**. Dùng nó cho `L7rerank`:

| lần | bài | điểm public |
|---|---|---|
| 1 | L1 thuần máy | **5,6** |
| 2 | L5 (QA VLM + 3 ghim) | **6,0** |
| 3 | **L7 = L5 + VLM rerank** | ? |

Chênh lệch giữa lần 3 và lần 2 **chính là giá trị của reranker**, đo trên đề
thật chứ không phải dev set tự làm. Đó là con số quyết định có mang nó vào đợt 1
tối nay hay không.

Nếu L7 > 6,0 → đợt 1 chạy `nop_bai.py` rồi `chay_rerank.py`, xong.
Nếu L7 ≤ 6,0 → bỏ reranker, quay lại bản L5, và biết chắc là không mất gì.

In [ ]:
!ls -la /kaggle/working/*.zip